> # ⚠️ ARCHIVED — DO NOT CITE ANY NUMBER FROM THIS NOTEBOOK
>
> This notebook is from the project's first generation (2026-08-11). It is kept
> to show the methodological path, **not** as evidence.
>
> - Its stored outputs have been **stripped**, deliberately, so no figure or table
>   here can be mistaken for a current result. Git history retains them.
> - It reads data paths and episode identifiers that **no longer exist**, so it
>   cannot be re-executed to regenerate them.
> - Where it uses the Grand Ouest reference, note that the reference has since been
>   re-resolved onto a new episode reconstruction, and the linkage methods,
>   thresholds, and splits all changed afterwards. Same source data, different
>   everything else.
>
> Current evidence lives in `notebooks/10`–`14`, the `*.md` reports at the
> repository root, and `reports/boamp_methodology_chapter.pdf`.


# Linkage Parameter Selection Diagnostics

This notebook checks which high-precision linkage parameters are reasonable before changing the renewal-linkage operating point. It uses the same candidate-pair evidence and the same high-precision gated scorer as notebook `08_high_precision_linkage_iteration.ipynb`.

The selection plots use **PILOT_DEVELOPMENT only**. `LOCKED_TEST` is not used to choose parameters.

## tl;dr

Run the notebook top-to-bottom to regenerate the parameter grid, tuning diagnostics, recommended pilot-only operating point, and exported figures under `data/processed/boamp_grand_ouest/figures/parameter_selection/`.

Because the pilot benchmark has only 16 anchors, these plots should guide parameter choice, not replace manual error review.

## Context & Methods

The model has two layers:

1. A weighted evidence score combining buyer, text, CPV/theme, time plausibility, and geography.
2. Acceptance gates that require strong buyer evidence, enough text similarity, CPV/theme continuity unless text is very high, and plausible time gap.

### Key Assumptions

- The manual `PILOT_DEVELOPMENT` split is the only split used to select parameters.
- A no-link decision is valid for anchors manually labeled as `NO_OBSERVED_SUCCESSOR_IN_SCOPE`.
- The diagnostic grid is intentionally small enough to be reproducible and inspectable.

In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid', context='notebook')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'boamp_grand_ouest'
FIGURE_DIR = PROCESSED_DIR / 'figures' / 'parameter_selection'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PAIRS_PATH = PROCESSED_DIR / 'renewal_candidate_pairs.parquet'
ANCHORS_PATH = PROCESSED_DIR / 'reference_anchor_episodes.parquet'
LINKS_PATH = PROCESSED_DIR / 'reference_successor_links.parquet'
HIGH_PRECISION_CONFIG_PATH = PROCESSED_DIR / 'high_precision_linkage_config.json'

GRID_OUTPUT_PATH = PROCESSED_DIR / 'parameter_selection_grid.csv'
RECOMMENDED_CONFIG_PATH = PROCESSED_DIR / 'parameter_selection_recommended_config.json'

PALETTE = {
    'blue': '#4E79A7',
    'gold': '#F2C14E',
    'orange': '#F28E2B',
    'pink': '#E15759',
    'olive': '#59A14F',
    'gray': '#6B7280',
    'ink': '#222222',
}

print('Project root:', PROJECT_ROOT)
print('Figure output:', FIGURE_DIR)

## Data

Load the candidate pairs and manual benchmark files. The benchmark is reduced to the eligible evaluation anchors, then split into `PILOT_DEVELOPMENT` and `LOCKED_TEST`.

In [ ]:
pairs = pd.read_parquet(PAIRS_PATH)
anchors = pd.read_parquet(ANCHORS_PATH)
links = pd.read_parquet(LINKS_PATH)
with HIGH_PRECISION_CONFIG_PATH.open('r', encoding='utf-8') as f:
    high_precision_config = json.load(f)
current_config = high_precision_config['selected_config']

required_pair_columns = [
    'sample_id', 'benchmark_split', 'candidate_idweb', 'buyer_match_type', 'buyer_name_similarity',
    'word_tfidf_similarity', 'char_ngram_tfidf_similarity', 'cpv_overlap_score', 'same_theme',
    'same_region', 'duration_gap_score', 'time_gap_days', 'time_gap_months'
]
missing_columns = [col for col in required_pair_columns if col not in pairs.columns]
assert not missing_columns, f'Missing required candidate-pair columns: {missing_columns}'

eligible_anchors = anchors[
    anchors['primary_evaluation_eligible'].fillna(False).astype(bool)
    & anchors['is_in_evaluation_subset'].fillna(False).astype(bool)
].copy()

eligible_links = links[links['is_primary_evaluation_eligible_anchor'].fillna(False).astype(bool)].copy()

print(f'Candidate pairs: {len(pairs):,}')
print(f'Eligible benchmark anchors: {len(eligible_anchors):,}')
print(eligible_anchors['benchmark_split'].value_counts().to_string())
print(f'Confirmed successor link rows for eligible anchors: {len(eligible_links):,}')
assert len(eligible_anchors) == 94, 'Expected 94 eligible anchors from the manual benchmark.'
assert set(eligible_anchors['benchmark_split'].dropna().unique()) == {'PILOT_DEVELOPMENT', 'LOCKED_TEST'}


### Build Truth Sets

For each anchor, collect the confirmed successor `idweb` values. Anchors without a confirmed successor become no-link negatives for benchmark evaluation.

In [ ]:
def parse_json_list(value) -> list[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(item) for item in value if pd.notna(item)]
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return []
    if isinstance(parsed, list):
        return [str(item) for item in parsed if pd.notna(item)]
    return []

truth_by_sample = {str(row.sample_id): set() for row in eligible_anchors.itertuples(index=False)}
for row in eligible_links.itertuples(index=False):
    sample_id = str(row.sample_id)
    truth_by_sample.setdefault(sample_id, set()).update(parse_json_list(row.successor_notice_ids_json))

truth = eligible_anchors[['sample_id', 'benchmark_split', 'final_outcome']].copy()
truth['sample_id'] = truth['sample_id'].astype(str)
truth['true_successor_idwebs'] = truth['sample_id'].map(lambda sample_id: sorted(truth_by_sample.get(sample_id, set())))
truth['has_true_successor'] = truth['true_successor_idwebs'].map(bool)

truth_summary = truth.groupby(['benchmark_split', 'has_true_successor']).size().unstack(fill_value=0)
display(truth_summary)
assert truth.loc[truth['benchmark_split'].eq('PILOT_DEVELOPMENT'), 'sample_id'].nunique() == 16
assert truth.loc[truth['benchmark_split'].eq('LOCKED_TEST'), 'sample_id'].nunique() == 78


## Scoring Functions

These functions intentionally mirror the high-precision iteration scorer. The only thing this notebook changes is the parameter grid.

In [ ]:
WEIGHTS = {
    'high_precision_v1': {'buyer': 0.40, 'text': 0.30, 'cpv': 0.15, 'time': 0.10, 'geo': 0.05},
    'text_precision_v1': {'buyer': 0.35, 'text': 0.40, 'cpv': 0.10, 'time': 0.10, 'geo': 0.05},
    'buyer_precision_v1': {'buyer': 0.50, 'text': 0.25, 'cpv': 0.15, 'time': 0.05, 'geo': 0.05},
}
BUYER_SCORE_MAP = {'siren': 1.0, 'normalized_name': 0.9, 'fuzzy_name': 0.7, 'token_overlap': 0.6}


def add_high_precision_components(frame: pd.DataFrame, weights_name: str) -> pd.DataFrame:
    tmp = frame.copy()
    weights = WEIGHTS[weights_name]
    tmp['hp_buyer_score'] = tmp['buyer_match_type'].map(BUYER_SCORE_MAP).fillna(0).astype(float)
    tmp['hp_text_score'] = tmp[['word_tfidf_similarity', 'char_ngram_tfidf_similarity']].max(axis=1).fillna(0).clip(0, 1)
    tmp['hp_cpv_score'] = tmp['cpv_overlap_score'].fillna(0).clip(0, 1)
    tmp['hp_time_score'] = tmp['duration_gap_score'].fillna(0).clip(0, 1)
    tmp['hp_geo_score'] = tmp['same_region'].astype(bool).astype(float)
    tmp['hp_strong_buyer'] = tmp['buyer_match_type'].isin(['siren', 'normalized_name']) | (
        tmp['buyer_match_type'].isin(['fuzzy_name', 'token_overlap']) & tmp['buyer_name_similarity'].fillna(0).ge(0.82)
    )
    tmp['hp_score'] = 100 * (
        weights['buyer'] * tmp['hp_buyer_score']
        + weights['text'] * tmp['hp_text_score']
        + weights['cpv'] * tmp['hp_cpv_score']
        + weights['time'] * tmp['hp_time_score']
        + weights['geo'] * tmp['hp_geo_score']
    )
    return tmp


def apply_high_precision_rules(frame: pd.DataFrame, config: dict) -> pd.DataFrame:
    tmp = add_high_precision_components(frame, config['weights'])
    base_gate = (
        tmp['hp_strong_buyer']
        & tmp['time_gap_days'].between(config['min_gap_days'], config['max_gap_days'])
        & tmp['hp_text_score'].ge(config['min_text_similarity'])
        & tmp['hp_time_score'].ge(config['min_time_score'])
    )
    cpv_gate = tmp['hp_cpv_score'].gt(0) | tmp['same_theme'].astype(bool) | tmp['hp_text_score'].ge(config['text_override_for_missing_cpv'])
    fuzzy_penalty_gate = ~tmp['buyer_match_type'].isin(['fuzzy_name', 'token_overlap']) | (
        tmp['hp_text_score'].ge(config['min_text_for_fuzzy_buyer'])
        & (tmp['hp_cpv_score'].gt(0) | tmp['same_theme'].astype(bool))
    )
    tmp['automatic_gate_pass'] = base_gate & cpv_gate & fuzzy_penalty_gate & tmp['hp_score'].ge(config['automatic_threshold'])
    return tmp


def predictions_from_scored(scored: pd.DataFrame, top_k: int = 5) -> dict[str, list[str]]:
    predictions = {sample_id: [] for sample_id in truth['sample_id']}
    accepted = scored[scored['automatic_gate_pass']].sort_values(
        ['sample_id', 'hp_score', 'hp_text_score', 'hp_cpv_score', 'time_gap_days'],
        ascending=[True, False, False, False, True],
    )
    for sample_id, group in accepted.groupby('sample_id'):
        predictions[str(sample_id)] = group['candidate_idweb'].astype(str).head(top_k).tolist()
    return predictions


def evaluate_predictions(predictions: dict[str, list[str]], split: str | None = None) -> dict:
    eval_truth = truth if split is None else truth[truth['benchmark_split'].eq(split)]
    rows = []
    for row in eval_truth.itertuples(index=False):
        predicted = predictions.get(row.sample_id, []) or []
        predicted = [str(x) for x in predicted]
        true_set = set(row.true_successor_idwebs)
        has_true = bool(true_set)
        top1 = predicted[0] if predicted else None
        top1_correct = bool(top1 and top1 in true_set)
        top5_correct = bool(true_set and any(candidate in true_set for candidate in predicted[:5]))
        rr = 0.0
        if true_set:
            for rank, candidate in enumerate(predicted, start=1):
                if candidate in true_set:
                    rr = 1.0 / rank
                    break
        rows.append({
            'sample_id': row.sample_id,
            'split': row.benchmark_split,
            'has_true_successor': has_true,
            'predicted_any': bool(predicted),
            'top1_correct': top1_correct,
            'top5_correct': top5_correct,
            'false_positive_no_successor': (not has_true and bool(predicted)),
            'no_link_correct': (not has_true and not predicted),
            'reciprocal_rank': rr,
        })
    frame = pd.DataFrame(rows)
    predicted_count = int(frame['predicted_any'].sum())
    positive_count = int(frame['has_true_successor'].sum())
    no_successor_count = int((~frame['has_true_successor']).sum())
    return {
        'anchors': int(len(frame)),
        'positive_anchors': positive_count,
        'no_successor_anchors': no_successor_count,
        'coverage_rate': float(frame['predicted_any'].mean()) if len(frame) else 0.0,
        'precision_at_1': float(frame['top1_correct'].sum() / predicted_count) if predicted_count else 0.0,
        'recall_at_1': float(frame['top1_correct'].sum() / positive_count) if positive_count else 0.0,
        'recall_at_5': float(frame['top5_correct'].sum() / positive_count) if positive_count else 0.0,
        'mean_reciprocal_rank': float(frame['reciprocal_rank'].mean()) if len(frame) else 0.0,
        'false_positive_rate_no_successor': float(frame['false_positive_no_successor'].sum() / no_successor_count) if no_successor_count else 0.0,
        'no_link_accuracy': float(frame['no_link_correct'].sum() / no_successor_count) if no_successor_count else 0.0,
        'predicted_count': predicted_count,
    }


def pilot_selection_objective(metrics: dict) -> float:
    if metrics['predicted_count'] == 0:
        return -1.0
    return (
        0.60 * metrics['precision_at_1']
        + 0.20 * metrics['no_link_accuracy']
        + 0.15 * metrics['recall_at_5']
        + 0.05 * metrics['coverage_rate']
    )


## Parameter Grid

The grid varies the operating knobs that directly control automatic-link strictness:

- `automatic_threshold`: minimum weighted evidence score.
- `min_text_similarity`: minimum text evidence required for any automatic link.
- `text_override_for_missing_cpv`: text evidence needed when CPV/theme continuity is weak.
- `min_time_score`: duration plausibility gate.
- `weights`: alternative evidence-weighting policies.

Only pilot metrics are computed in this grid.

In [ ]:
threshold_values = list(range(50, 91, 5))
text_similarity_values = [round(x, 2) for x in np.arange(0.10, 0.61, 0.05)]
text_override_values = [0.45, 0.55, 0.65]
min_time_score_values = [0.0, 0.15, 0.30]
weight_names = list(WEIGHTS)

config_grid = []
for weights in weight_names:
    for automatic_threshold in threshold_values:
        for min_text_similarity in text_similarity_values:
            for text_override_for_missing_cpv in text_override_values:
                for min_time_score in min_time_score_values:
                    config_grid.append({
                        'weights': weights,
                        'automatic_threshold': automatic_threshold,
                        'min_text_similarity': min_text_similarity,
                        'text_override_for_missing_cpv': text_override_for_missing_cpv,
                        'min_text_for_fuzzy_buyer': max(0.45, round(min_text_similarity + 0.10, 2)),
                        'min_time_score': min_time_score,
                        'min_gap_days': 180,
                        'max_gap_days': 8 * 365,
                        'manual_review_threshold': 45,
                        'manual_review_min_text': 0.10,
                        'top_k': 5,
                    })

print(f'Grid size: {len(config_grid):,} configurations')

In [ ]:
grid_records = []
for config_id, config in enumerate(config_grid):
    scored = apply_high_precision_rules(pairs, config)
    predictions = predictions_from_scored(scored, top_k=config['top_k'])
    metrics = evaluate_predictions(predictions, split='PILOT_DEVELOPMENT')
    grid_records.append({
        'config_id': config_id,
        **{k: config[k] for k in [
            'weights', 'automatic_threshold', 'min_text_similarity', 'text_override_for_missing_cpv',
            'min_text_for_fuzzy_buyer', 'min_time_score', 'min_gap_days', 'max_gap_days', 'top_k'
        ]},
        **metrics,
        'objective': pilot_selection_objective(metrics),
        'config_json': json.dumps(config, ensure_ascii=False),
    })

selection_grid = pd.DataFrame(grid_records)
selection_grid.to_csv(GRID_OUTPUT_PATH, index=False, encoding='utf-8')
print(f'Saved grid: {GRID_OUTPUT_PATH}')
display(selection_grid.sort_values(['objective', 'precision_at_1', 'recall_at_5', 'coverage_rate'], ascending=False).head(15))

## Recommendation Logic

For this stage, the main goal is high precision, but the chosen setting should not be a trivial one-link configuration. The recommendation therefore requires at least three automatic pilot predictions and then maximizes a precision-oriented objective.

In [ ]:
current_mask = pd.Series(True, index=selection_grid.index)
for key in ['weights', 'automatic_threshold', 'min_text_similarity', 'text_override_for_missing_cpv', 'min_time_score']:
    current_mask &= selection_grid[key].eq(current_config[key])
current_grid_row = selection_grid[current_mask].copy()
assert len(current_grid_row) == 1, 'Current high-precision config should appear exactly once in the diagnostic grid.'
current_grid_row = current_grid_row.iloc[0]

eligible_for_recommendation = selection_grid[selection_grid['predicted_count'].ge(3)].copy()
recommended_row = eligible_for_recommendation.sort_values(
    ['objective', 'precision_at_1', 'no_link_accuracy', 'recall_at_5', 'coverage_rate', 'automatic_threshold'],
    ascending=[False, False, False, False, False, False],
).iloc[0]
recommended_config = json.loads(recommended_row['config_json'])

recommendation_payload = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'selection_split': 'PILOT_DEVELOPMENT',
    'locked_test_used_for_selection': False,
    'selection_note': 'Pilot-only diagnostic recommendation. Do not replace the frozen operating point without reviewing error cases and then evaluating once on LOCKED_TEST.',
    'minimum_predicted_count_constraint': 3,
    'current_high_precision_config': current_config,
    'current_pilot_metrics': {k: (float(current_grid_row[k]) if isinstance(current_grid_row[k], (np.floating, float)) else int(current_grid_row[k]) if isinstance(current_grid_row[k], (np.integer, int)) else current_grid_row[k]) for k in ['config_id', 'anchors', 'positive_anchors', 'no_successor_anchors', 'coverage_rate', 'precision_at_1', 'recall_at_1', 'recall_at_5', 'mean_reciprocal_rank', 'false_positive_rate_no_successor', 'no_link_accuracy', 'predicted_count', 'objective']},
    'recommended_config_from_grid': recommended_config,
    'recommended_pilot_metrics': {k: (float(recommended_row[k]) if isinstance(recommended_row[k], (np.floating, float)) else int(recommended_row[k]) if isinstance(recommended_row[k], (np.integer, int)) else recommended_row[k]) for k in ['config_id', 'anchors', 'positive_anchors', 'no_successor_anchors', 'coverage_rate', 'precision_at_1', 'recall_at_1', 'recall_at_5', 'mean_reciprocal_rank', 'false_positive_rate_no_successor', 'no_link_accuracy', 'predicted_count', 'objective']},
}
with RECOMMENDED_CONFIG_PATH.open('w', encoding='utf-8') as f:
    json.dump(recommendation_payload, f, indent=2, ensure_ascii=False)

comparison = pd.DataFrame([
    {'setting': 'current_high_precision_config', **recommendation_payload['current_pilot_metrics']},
    {'setting': 'recommended_from_pilot_grid', **recommendation_payload['recommended_pilot_metrics']},
])
print(f'Saved recommendation: {RECOMMENDED_CONFIG_PATH}')
display(comparison[['setting', 'precision_at_1', 'recall_at_5', 'false_positive_rate_no_successor', 'no_link_accuracy', 'coverage_rate', 'predicted_count', 'objective']])
print(json.dumps(recommended_config, indent=2, ensure_ascii=False))

## Visual Diagnostics

All plots below are based on `PILOT_DEVELOPMENT` only. They are designed to show where precision improves, where coverage collapses, and where false positives become unacceptable.

In [ ]:
def save_current_figure(filename: str) -> Path:
    path = FIGURE_DIR / filename
    plt.savefig(path, dpi=170, bbox_inches='tight', facecolor='white')
    return path

# Use the current weight/override/time setting for two-dimensional heatmaps, so the heatmaps isolate the two most interpretable knobs.
focused_grid = selection_grid[
    selection_grid['weights'].eq(current_config['weights'])
    & selection_grid['text_override_for_missing_cpv'].eq(current_config['text_override_for_missing_cpv'])
    & selection_grid['min_time_score'].eq(current_config['min_time_score'])
].copy()

figure_manifest = []

curve_text_values = [0.10, 0.20, 0.25, 0.30, 0.40, 0.50]
curve_data = focused_grid[focused_grid['min_text_similarity'].isin(curve_text_values)].copy()
curve_long = curve_data.melt(
    id_vars=['automatic_threshold', 'min_text_similarity'],
    value_vars=['precision_at_1', 'recall_at_5', 'coverage_rate', 'false_positive_rate_no_successor'],
    var_name='metric',
    value_name='value',
)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=False)
for ax, metric in zip(axes.flat, ['precision_at_1', 'recall_at_5', 'coverage_rate', 'false_positive_rate_no_successor']):
    plot_data = curve_long[curve_long['metric'].eq(metric)]
    sns.lineplot(data=plot_data, x='automatic_threshold', y='value', hue='min_text_similarity', marker='o', ax=ax, palette='viridis')
    ax.axvline(current_config['automatic_threshold'], color=PALETTE['pink'], linestyle='--', linewidth=1.2)
    ax.set_title(metric.replace('_', ' '))
    ax.set_xlabel('Automatic score threshold')
    ax.set_ylabel('Metric value')
    ax.set_ylim(-0.03, 1.03)
    ax.legend(title='Min text', fontsize=8, title_fontsize=8)
fig.suptitle('Pilot metric curves by score threshold and minimum text similarity', y=1.02, fontsize=14)
fig.tight_layout()
path = save_current_figure('01_pilot_threshold_metric_curves.png')
figure_manifest.append(['pilot_threshold_metric_curves', str(path)])
plt.show()

frontier = selection_grid.copy()
fig, ax = plt.subplots(figsize=(9.5, 6.2))
sns.scatterplot(
    data=frontier,
    x='coverage_rate',
    y='precision_at_1',
    hue='false_positive_rate_no_successor',
    size='recall_at_5',
    style='weights',
    sizes=(35, 260),
    palette='rocket_r',
    alpha=0.80,
    ax=ax,
)
ax.scatter(current_grid_row['coverage_rate'], current_grid_row['precision_at_1'], s=260, marker='X', color=PALETTE['blue'], edgecolor='white', linewidth=1.2, label='current config')
ax.scatter(recommended_row['coverage_rate'], recommended_row['precision_at_1'], s=260, marker='*', color=PALETTE['gold'], edgecolor=PALETTE['ink'], linewidth=0.8, label='recommended')
ax.set_title('Pilot precision-coverage frontier')
ax.set_xlabel('Coverage rate')
ax.set_ylabel('Precision@1')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
fig.tight_layout()
path = save_current_figure('02_pilot_precision_coverage_frontier.png')
figure_manifest.append(['pilot_precision_coverage_frontier', str(path)])
plt.show()


In [ ]:
def heatmap_for_metric(metric: str, title: str, filename: str, cmap: str, vmin: float = 0.0, vmax: float = 1.0) -> Path:
    pivot = focused_grid.pivot_table(
        index='min_text_similarity',
        columns='automatic_threshold',
        values=metric,
        aggfunc='mean',
    ).sort_index(ascending=False)
    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.2f',
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidths=0.5,
        linecolor='white',
        cbar_kws={'label': metric.replace('_', ' ')},
        ax=ax,
    )
    ax.scatter(
        [list(pivot.columns).index(current_config['automatic_threshold']) + 0.5],
        [list(pivot.index).index(current_config['min_text_similarity']) + 0.5],
        marker='X', s=180, color=PALETTE['pink'], edgecolor='white', linewidth=1.0, label='current config'
    )
    if recommended_config['weights'] == current_config['weights'] and recommended_config['text_override_for_missing_cpv'] == current_config['text_override_for_missing_cpv'] and recommended_config['min_time_score'] == current_config['min_time_score']:
        ax.scatter(
            [list(pivot.columns).index(recommended_config['automatic_threshold']) + 0.5],
            [list(pivot.index).index(recommended_config['min_text_similarity']) + 0.5],
            marker='*', s=220, color=PALETTE['gold'], edgecolor=PALETTE['ink'], linewidth=0.8, label='recommended'
        )
    ax.set_title(title)
    ax.set_xlabel('Automatic score threshold')
    ax.set_ylabel('Minimum text similarity')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=True)
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    path = save_current_figure(filename)
    figure_manifest.append([filename.replace('.png', ''), str(path)])
    plt.show()
    return path

heatmap_for_metric('precision_at_1', 'Pilot precision@1 by threshold and text gate', '03_pilot_precision_heatmap_threshold_text.png', 'Blues')
heatmap_for_metric('false_positive_rate_no_successor', 'Pilot false-positive rate by threshold and text gate', '04_pilot_false_positive_heatmap_threshold_text.png', 'Reds')
heatmap_for_metric('recall_at_5', 'Pilot recall@5 by threshold and text gate', '05_pilot_recall5_heatmap_threshold_text.png', 'Greens')


In [ ]:
comparison_long = comparison.melt(
    id_vars=['setting'],
    value_vars=['precision_at_1', 'recall_at_5', 'false_positive_rate_no_successor', 'no_link_accuracy', 'coverage_rate'],
    var_name='metric',
    value_name='value',
)
fig, ax = plt.subplots(figsize=(11.5, 5.8))
sns.barplot(data=comparison_long, x='metric', y='value', hue='setting', palette=[PALETTE['blue'], PALETTE['gold']], ax=ax)
ax.set_title('Pilot metric comparison: current setting vs grid recommendation')
ax.set_xlabel('Metric')
ax.set_ylabel('Value')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=20)
ax.legend(title='Setting', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
path = save_current_figure('06_pilot_current_vs_recommended_metrics.png')
figure_manifest.append(['pilot_current_vs_recommended_metrics', str(path)])
plt.show()

figure_manifest_df = pd.DataFrame(figure_manifest, columns=['figure', 'path'])
display(figure_manifest_df)


## Takeaways

Use these diagnostics as the parameter-selection evidence:

- The heatmaps show which combinations of `automatic_threshold` and `min_text_similarity` protect precision and no-link accuracy.
- The frontier plot shows the tradeoff between linking rate and precision.
- The current high-precision setting is marked with `X`; the pilot-grid recommendation is marked with a star when it appears in the focused heatmap.
- Do not tune on `LOCKED_TEST`; after choosing a single setting from these plots, evaluate it once on `LOCKED_TEST` and then inspect errors manually.